# Bonds and fixed income

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`. Price **vanilla fixed**, a **floating (FRN-style) bond**, and an **amortizing** bond; request rate risk and spread-style metrics where the registry exposes them.


## Concept

- **Vanilla fixed:** PV from discount curve + schedule.
- **FRN:** floating leg needs a **forward curve** for projected index fixings.
- **Amortizing:** principal path feeds coupon bases.
- **Risk:** `dv01`, **modified duration**, **convexity**; **Z-spread** / **I-spread** appear when the pricer stack supports them for your spec.


In [1]:
import json
from datetime import date

from finstack_quant.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext, HazardCurve

print("Imports OK (bonds).")


Imports OK (bonds).


## Instrument JSON


In [2]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/bonds_and_fixed_income.json").read_text())

AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

fixed_bond = _NOTEBOOK_DATA['fixed_bond']

frn_bond = _NOTEBOOK_DATA['frn_bond']

amort_bond = _NOTEBOOK_DATA['amort_bond']

for label, payload in (("fixed", fixed_bond), ("frn", frn_bond), ("amort", amort_bond)):
    try:
        validate_instrument_json(json.dumps(payload))
        print(label, "JSON OK")
    except Exception as exc:
        print(label, "validate:", type(exc).__name__, exc)

bond_fixed_json = json.dumps(fixed_bond)
bond_frn_json = json.dumps(frn_bond)
bond_amort_json = json.dumps(amort_bond)


fixed JSON OK
frn JSON OK
amort JSON OK


## MarketContext


In [3]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
    base_date=AS_OF,
    day_count="act_360",
)
mc = MarketContext().insert(ois).insert(sofr)
market_json = mc.to_json()
print("curves in snapshot:", len(json.loads(market_json)["curves"]))


curves in snapshot: 2


## Pricing


In [4]:
for label, bjson in (
    ("fixed", bond_fixed_json),
    ("frn", bond_frn_json),
    ("amort", bond_amort_json),
):
    try:
        out = price_instrument_with_metrics(bjson, market_json, AS_OF_STR, model="discounting")
        print(label, ValuationResult.from_json(out))
    except Exception as exc:
        print(label, "price:", type(exc).__name__, exc)


fixed ValuationResult(id="UST-FIXED-10Y", price=987618.0556, currency=USD, metrics=0)
frn price: ValueError Calibration error: Validation error: floating-rate observation date 2024-01-15 is before the 'USD-SOFR-3M' curve base date 2025-01-15; the realized historical fixings are missing — provide a MarketContext ScalarTimeSeries with id 'FIXING:USD-SOFR-3M', supply a curve based on/before the observation date, or configure a FloatingRateFallback [instrument=FRN-USD-SOFR, type=Bond, model=Discounting]
amort ValuationResult(id="AMORT-NOTE", price=872411.3348, currency=USD, metrics=0)


## Metrics


In [5]:
metrics_req = ["dv01", "duration_mod", "convexity", "z_spread", "i_spread"]
for label, bjson in (("fixed", bond_fixed_json), ("frn", bond_frn_json)):
    try:
        out = price_instrument_with_metrics(
            bjson, market_json, AS_OF_STR, model="discounting", metrics=metrics_req
        )
        vr = ValuationResult.from_json(out)
        print("—", label)
        for m in metrics_req:
            v = vr.get_metric(m)
            if v is not None:
                print(f"  {m}: {v}")
    except Exception as exc:
        print(label, "metrics:", type(exc).__name__, exc)


— fixed
  dv01: -730.542811110151
  duration_mod: 7.397017571739839
  convexity: 0.6463863062536874
  z_spread: 0.0015566814876780135
  i_spread: 0.0015558094790232152
frn metrics: ValueError Calibration error: Validation error: floating-rate observation date 2024-01-15 is before the 'USD-SOFR-3M' curve base date 2025-01-15; the realized historical fixings are missing — provide a MarketContext ScalarTimeSeries with id 'FIXING:USD-SOFR-3M', supply a curve based on/before the observation date, or configure a FloatingRateFallback [instrument=FRN-USD-SOFR, type=Bond, model=Discounting]


## Price-from-metric overrides

`PricingOverrides.market_quotes` supports **nine mutually-exclusive price-driving fields** that short-circuit model pricing with a market quote. At most one may be set per instrument; `validate` rejects multiple.

| Field                      | Units                              | Driven through                  |
|----------------------------|------------------------------------|---------------------------------|
| `quoted_clean_price`       | % of par (e.g. `98.5`)             | accrued → dirty                 |
| `quoted_dirty_price_ccy`   | currency units                     | direct                          |
| `quoted_ytm`               | decimal (e.g. `0.055` = 5.5%)      | street-convention YTM inversion |
| `quoted_ytw`               | decimal                            | maturity-flow YTM (use OAS for callable exercise) |
| `quoted_z_spread`          | decimal (e.g. `0.0125` = 125bp)    | Z-spread over discount curve    |
| `quoted_oas`               | decimal                            | OAS over short-rate tree        |
| `quoted_discount_margin`   | decimal (FRNs)                     | additive margin on float leg    |
| `quoted_i_spread`          | decimal                            | YTM − par-swap-rate             |
| `quoted_asw_market`        | decimal                            | market-convention ASW inversion |

The legacy `quoted_spread_bp` field has been renamed to `cds_quote_bp` (CDS-only); the old JSON key is still accepted as a serde alias.

In [6]:
import copy


def price_with_override(overrides: dict) -> float:
    spec = copy.deepcopy(fixed_bond)
    spec["spec"]["pricing_overrides"] = overrides
    out = price_instrument_with_metrics(
        json.dumps(spec), market_json, AS_OF_STR, model="discounting"
    )
    return ValuationResult.from_json(out).price


# Baseline: no override, pure model PV.
baseline_bond = copy.deepcopy(fixed_bond)
baseline_bond["spec"]["pricing_overrides"] = {}
baseline = price_with_override({})
print(f"baseline model PV               : {baseline:,.2f} USD")

# 1) Override by quoted clean price (already used above with 98.75).
clean_pv = price_with_override({"quoted_clean_price": 99.5})
print(f"quoted_clean_price=99.5         : {clean_pv:,.2f} USD")

# 2) Override by quoted dirty price (currency units).
dirty_pv = price_with_override({"quoted_dirty_price_ccy": 985_000.00})
print(f"quoted_dirty_price_ccy=985000   : {dirty_pv:,.2f} USD")

# 3) Override by YTM (decimal).
ytm_pv = price_with_override({"quoted_ytm": 0.055})
print(f"quoted_ytm=0.055                : {ytm_pv:,.2f} USD")

# 4) Override by Z-spread (decimal, additive over the discount curve).
zspread_pv = price_with_override({"quoted_z_spread": 0.0075})
print(f"quoted_z_spread=0.0075 (75bp)   : {zspread_pv:,.2f} USD")

# 5) Override by I-spread.
ispread_pv = price_with_override({"quoted_i_spread": 0.0050})
print(f"quoted_i_spread=0.0050 (50bp)   : {ispread_pv:,.2f} USD")

# 6) Mutual-exclusivity check: setting two price drivers is rejected by validate.
try:
    validate_instrument_json(
        json.dumps(
            {
                **fixed_bond,
                "spec": {
                    **fixed_bond["spec"],
                    "pricing_overrides": {"quoted_ytm": 0.05, "quoted_z_spread": 0.01},
                },
            }
        )
    )
except Exception as exc:
    print(f"\nmutual-exclusivity check rejects dual drivers: {type(exc).__name__}")

baseline model PV               : 998,960.99 USD
quoted_clean_price=99.5         : 995,118.06 USD
quoted_dirty_price_ccy=985000   : 985,000.00 USD
quoted_ytm=0.055                : 912,220.84 USD
quoted_z_spread=0.0075 (75bp)   : 945,397.28 USD
quoted_i_spread=0.0050 (50bp)   : 962,831.32 USD

mutual-exclusivity check rejects dual drivers: ValueError


## Scenario price shocks

`scenario_price_shock_pct` applies a **multiplicative PV shock** on top of whatever pricing path was taken (model PV or quoted-price override). The engine applies the shock **exactly once** regardless of whether you call `value()` directly or go through `price_with_metrics`. This is the canonical lever for downside/upside revaluations in portfolio stress tests.

In [7]:
shock_pct = -0.10  # -10% price shock
shocked_pv = price_with_override(
    {"quoted_clean_price": 98.75, "scenario_price_shock_pct": shock_pct}
)
unshocked_pv = price_with_override({"quoted_clean_price": 98.75})
print(f"unshocked PV                    : {unshocked_pv:,.2f} USD")
print(f"shocked PV (-10%)               : {shocked_pv:,.2f} USD")
print(f"ratio (should be ~0.90)         : {shocked_pv / unshocked_pv:.6f}")

# Scenario shock composes with any price-from-metric override: here we pin the
# bond at a 75bp Z-spread and apply a +5% upside shock on top.
combo_pv = price_with_override(
    {"quoted_z_spread": 0.0075, "scenario_price_shock_pct": 0.05}
)
base_zspread_pv = price_with_override({"quoted_z_spread": 0.0075})
print(f"\nZ-spread PV                     : {base_zspread_pv:,.2f} USD")
print(f"Z-spread PV + 5% upside shock   : {combo_pv:,.2f} USD")
print(f"ratio (should be 1.05)          : {combo_pv / base_zspread_pv:.6f}")

unshocked PV                    : 987,618.06 USD
shocked PV (-10%)               : 888,856.25 USD
ratio (should be ~0.90)         : 0.900000

Z-spread PV                     : 945,397.28 USD
Z-spread PV + 5% upside shock   : 992,667.15 USD
ratio (should be 1.05)          : 1.050000


## Takeaways

- **One pipeline** prices fixed, float, and amortizing bonds once curves are wired (`USD-OIS` + `USD-SOFR-3M` for FRNs).
- **Metrics** are registry-driven: missing names are omitted rather than erroring.
- **Quoted clean** on the fixed bond pins a level so the engine can imply yields/spreads consistently with desk practice.
- **Nine price-driving overrides** (`quoted_clean_price`, `quoted_dirty_price_ccy`, `quoted_ytm/ytw`, `quoted_z_spread`, `quoted_oas`, `quoted_discount_margin`, `quoted_i_spread`, `quoted_asw_market`) short-circuit model pricing; validation rejects setting more than one at a time.
- **`scenario_price_shock_pct`** applies a multiplicative PV shock exactly once regardless of entry point, and composes cleanly with any quoted-price override.

## Which bond pricing model to use?

| Situation                              | Use model=      | Key data required                     | Notes |
|----------------------------------------|-----------------|---------------------------------------|-------|
| Vanilla / FRN / amortizing, no credit, no calls | discounting (default) | DiscountCurve | Fast PV + standard risk metrics |
| Credit-risky (hazard / reduced form)   | hazard_rate     | DiscountCurve + HazardCurve + bond.credit_curve_id | Survival-weighted + FRP recovery |
| Embedded options (callable/putable), OAS | tree            | DiscountCurve (+Hazard optional), quoted_clean_price, call_put schedule | Uses short-rate tree (Ho-Lee/HW/BDT) |
| PIK / toggle bonds, structural credit view | merton_mc     | DiscountCurve, merton_mc_config on overrides (MertonModel + schedule) | Monte Carlo with asset diffusion, first-passage default, PIK rules |

All paths are available via `price_instrument_with_metrics(..., model=...)`.


## Credit-risky bonds (hazard rate)

Use `model="hazard_rate"` + an explicit `credit_curve_id` on the bond + a `HazardCurve` in the market.

The engine applies:
- Survival weighting on coupons and principal (alive leg)
- Fractional recovery of par (FRP) on the outstanding notional at default time

Higher hazard → lower price. Higher recovery → higher price. CS01 becomes available.


In [8]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/bonds_and_fixed_income.json").read_text())

hazard_bond = _NOTEBOOK_DATA['hazard_bond']
validate_instrument_json(json.dumps(hazard_bond))
bond_hazard_json = json.dumps(hazard_bond)


In [9]:
hazard = HazardCurve(
    "USD-CREDIT",
    AS_OF,
    [(0.0, 0.02), (5.0, 0.025), (10.0, 0.03)],
    recovery_rate=0.40,
)
mc_haz = mc.insert(hazard)  # reuse the earlier ois + sofr mc
market_haz_json = mc_haz.to_json()
print("market has hazard:", "USD-CREDIT" in json.loads(market_haz_json).get("hazard_curves", {}))


market has hazard: False


In [10]:
# Price with hazard (ensure import)
from finstack_quant.core.market_data import HazardCurve

# Rebuild a clean mc for this section to avoid scope issues
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
hazard = HazardCurve(
    "USD-CREDIT",
    AS_OF,
    [(0.0, 0.02), (5.0, 0.025), (10.0, 0.03)],
    recovery_rate=0.40,
)
mc_haz = MarketContext().insert(ois).insert(hazard)
market_haz_json = mc_haz.to_json()

out = price_instrument_with_metrics(
    bond_hazard_json, market_haz_json, AS_OF_STR, model="hazard_rate"
)
print("hazard price:", ValuationResult.from_json(out))


hazard price: ValuationResult(id="CORP-HZD-5Y", price=1017464.6769, currency=USD, metrics=0)


In [11]:
# Metrics under hazard (CS01 etc.)
metrics_haz = ["cs01", "bucketed_cs01"]
out = price_instrument_with_metrics(
    bond_hazard_json, market_haz_json, AS_OF_STR, model="hazard_rate", metrics=metrics_haz
)
vr = ValuationResult.from_json(out)
print("hazard metrics present:", [m for m in metrics_haz if vr.get_metric(m) is not None])
print("cs01:", vr.get_metric("cs01"))


hazard metrics present: ['cs01', 'bucketed_cs01']
cs01: -221.7281580833951


In [12]:
# Effect of hazard level (same bond, two curves)
h_low = HazardCurve("USD-CREDIT-LOW", AS_OF, [(0.0, 0.01), (5.0, 0.01)], recovery_rate=0.40)
h_high = HazardCurve("USD-CREDIT-HI", AS_OF, [(0.0, 0.05), (5.0, 0.05)], recovery_rate=0.40)
m_low = MarketContext().insert(ois).insert(h_low)
m_high = MarketContext().insert(ois).insert(h_high)

bond_lowh = json.loads(bond_hazard_json)
bond_lowh["spec"]["credit_curve_id"] = "USD-CREDIT-LOW"
bond_highh = json.loads(bond_hazard_json)
bond_highh["spec"]["credit_curve_id"] = "USD-CREDIT-HI"

pv_low = ValuationResult.from_json(
    price_instrument_with_metrics(json.dumps(bond_lowh), m_low.to_json(), AS_OF_STR, model="hazard_rate")
).price
pv_high = ValuationResult.from_json(
    price_instrument_with_metrics(json.dumps(bond_highh), m_high.to_json(), AS_OF_STR, model="hazard_rate")
).price
print(f"low hazard (1%) PV: {pv_low:,.2f}")
print(f"high hazard (5%) PV: {pv_high:,.2f}")
print("higher hazard produces lower price:", pv_high < pv_low)


low hazard (1%) PV: 1,040,067.94
high hazard (5%) PV: 954,624.40
higher hazard produces lower price: True


## Callable bonds and OAS (tree)

Bonds with `call_put` schedules use a short-rate tree for PV (and for OAS).

- Explicit `model="tree"` + `quoted_clean_price` lets you back out OAS.
- Tree model variants (Ho-Lee, Hull-White, BDT) are selected via `pricing_overrides.model_config`:
  - default (no call) or no vol_model → Ho-Lee
  - with call_put, no Black → Hull-White using mean_reversion + implied_volatility (or vol)
  - vol_model: "black" + implied_volatility → Black-Derman-Toy (lognormal)
- Set `tree_steps` to control accuracy.

For this demo we price a simple callable via the tree path and request the OAS metric.


In [13]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/bonds_and_fixed_income.json").read_text())

callable_bond = _NOTEBOOK_DATA['callable_bond']
validate_instrument_json(json.dumps(callable_bond))
callable_json = json.dumps(callable_bond)


In [14]:
# Price via tree (OAS path) and request oas metric. Reuse simple ois market.
out = price_instrument_with_metrics(
    callable_json, market_json, AS_OF_STR, model="tree", metrics=["oas"]
)
vr = ValuationResult.from_json(out)
print("tree PV (callable):", vr.price)
print("oas_bp:", vr.get_metric("oas_bp") or vr.get_metric("oas"))


tree PV (callable): 990138.888888889
oas_bp: 134.65554649016116


In [15]:
import copy

def price_callable_with_overrides(ov):
    spec = copy.deepcopy(callable_bond)
    po = spec["spec"].setdefault("pricing_overrides", {})
    # Due to serde(flatten) on ModelConfig, these live directly under pricing_overrides in JSON
    for k in ("vol_model", "mean_reversion", "tree_steps"):
        if k in ov:
            po[k] = ov[k]
    if "implied_volatility" in ov:
        po["implied_volatility"] = ov["implied_volatility"]
    if "quoted_clean_price" not in po:
        po["quoted_clean_price"] = 99.0
    out = price_instrument_with_metrics(
        json.dumps(spec), market_json, AS_OF_STR, model="tree", metrics=["oas"]
    )
    return ValuationResult.from_json(out)

print("HW-like (mean rev):", price_callable_with_overrides({
    "mean_reversion": 0.03,
    "implied_volatility": 0.01,
    "tree_steps": 50
}).get_metric("oas_bp"))

# BDT via vol_model + implied vol (lognormal)
print("BDT-like:", price_callable_with_overrides({
    "vol_model": "black",
    "implied_volatility": 0.20,
    "tree_steps": 50
}).get_metric("oas_bp"))


HW-like (mean rev): 134.44071927372215
BDT-like: 139.66250564095506


## PIK bonds with structural credit (Merton MC)

`model="merton_mc"` prices PIK/toggle bonds using a Merton structural model + Monte Carlo.

You supply a `MertonMcConfig` (asset value/vol/barrier/rate, pik schedule, optional endogenous hazard, dynamic recovery, toggle rule, paths/seed).

In Python/JSON, put the `merton_mc_config` object directly under `pricing_overrides` (it is part of the flattened model config section, like `tree_steps` or `vol_model`).

The example below uses a simple PIK schedule and modest paths for speed.


In [16]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/bonds_and_fixed_income.json").read_text())

# PIK bond JSON + merton_mc_config (serializable now)
pik_bond = _NOTEBOOK_DATA['pik_bond']
validate_instrument_json(json.dumps(pik_bond))
pik_json = json.dumps(pik_bond)


In [17]:
# Price the PIK bond with merton_mc (re-use ois from earlier; build minimal mc)
out = price_instrument_with_metrics(
    pik_json, market_json, AS_OF_STR, model="merton_mc"
)
vr = ValuationResult.from_json(out)
print("merton_mc price (pct of notional ~):", round(vr.price / 100.0 , 4) * 100 , "% of par (notional=100)")
print("expected_loss:", vr.get_metric("expected_loss"))
print("default_rate:", vr.get_metric("default_rate"))
print("mc_stderr:", vr.get_metric("mc_stderr"))


merton_mc price (pct of notional ~): 107.14999999999999 % of par (notional=100)
expected_loss: 0.08951980368242651
default_rate: 0.1425
mc_stderr: 0.5281091950468558


In [18]:
# Quick sanity: a cash coupon version of same params should be higher price than full PIK under risk
cash_equiv = json.loads(pik_json)
cash_equiv["spec"]["cashflow_spec"]["Fixed"]["coupon_type"] = "Cash"
cash_json = json.dumps(cash_equiv)

vr_cash = ValuationResult.from_json( price_instrument_with_metrics(cash_json, market_json, AS_OF_STR, model="merton_mc") )
vr_pik = vr
print("cash equiv price:", round(vr_cash.price, 4))
print("pik price:", round(vr_pik.price, 4))
print("pik lower than cash (risk + accrual effect):", vr_pik.price < vr_cash.price)


cash equiv price: 107.1519
pik price: 107.1519
pik lower than cash (risk + accrual effect): False


## Takeaways (all bond models)

- **discounting**: default risk-free DF cashflow PV. Use for plain bonds.
- **hazard_rate**: opt-in credit via `credit_curve_id` + `HazardCurve`. Produces survival-weighted PV + FRP recovery; enables CS01.
- **tree**: auto for bonds with `call_put` schedules (or explicit `model="tree"` with quote). Supports Ho-Lee / Hull-White / BDT; OAS via quoted price + tree.
- **merton_mc**: structural MC for PIK/toggle bonds. Provide `merton_mc_config` (MertonModel + PikSchedule + options); returns expected loss, default stats, SE, etc.
- All models participate in the same `price_instrument*` + metrics registry surface. Missing metrics are omitted.
- Price-driving overrides (quoted_*) and `scenario_price_shock_pct` compose with the chosen model.
